# Classifying Players — Multinomial Logistic, Step by Step

**This notebook is the runnable twin of [`multinomial-walkthrough.html`](multinomial-walkthrough.html).** Same 12 steps, but you can press *Run* and watch each one happen on our real player data.

**How to read it:** each step has a short *plain-English* note (what + why), then a clean code cell. You don't need to understand the code to follow along — the notes carry the story. If you *do* read the code, it stays short and references the note above it.

**Related:** [Binary logistic notebook](ovr-notebook.ipynb) · [MN vs OvR explainer](mn-vs-ovr.html)

---

### One-time setup
Run this if the libraries aren't installed yet.

In [ ]:
# Setup — install the tools we use (safe to re-run; skips if already present)
# !pip install pandas scikit-learn matplotlib statsmodels

In [ ]:
# Imports + find the data file
import json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# The data lives in the repo's data/ folder. Try a few likely locations.
CANDIDATES = [Path('../../data/players.json'), Path('data/players.json'), Path('players.json')]
DATA_PATH = next((p for p in CANDIDATES if p.exists()), None)
assert DATA_PATH, 'Could not find players.json — set DATA_PATH manually.'
print('Using:', DATA_PATH.resolve())

## Step 0 — Build the answer column

**What:** our data ships the *clues* about each player but not their *type* — the model needs an "answer" column to learn from. The types are documented as ID-range rules in the file's notes (e.g. players P-176–P-183 are `aggressive_predatory`). We turn those rules into a label per player.

**Why:** a model can only learn the pattern if we show it the right answers for the study examples. This cell creates them.

*(Nice side effect: because the types are essentially defined by the clues, the model should later "rediscover" sensible reasons — a satisfying check that it learned the right thing.)*

In [ ]:
# Step 0 — load players and derive the archetype label from the documented ID ranges
raw = json.loads(DATA_PATH.read_text(encoding='utf-8'))
players = raw['players'] if isinstance(raw, dict) and 'players' in raw else raw
df = pd.DataFrame(players)

def archetype_from_id(pid):
    n = int(str(pid).split('-')[1])           # 'P-176' -> 176
    if   100 <= n <= 107: return 'new'
    elif 108 <= n <= 141: return 'recreational'
    elif 142 <= n <= 163: return 'regular'
    elif 164 <= n <= 175: return 'grinder'
    elif 176 <= n <= 183: return 'aggressive_predatory'
    elif 184 <= n <= 191: return 'promo_hunter'
    elif 192 <= n <= 197: return 'shared_device_household'
    elif 198 <= n <= 202: return 'cluster_member'
    elif 203 <= n <= 220: return 'healthy_anchor'
    else:                 return 'bot_like'      # P-221

df['archetype'] = df['player_id'].map(archetype_from_id)
print(f'{len(df)} players, {df.archetype.nunique()} types')
df['archetype'].value_counts()

## Step 1 — The goal

**What:** for every player, predict which of 9 types they are, from their stats.

**Why:** the rest of the system needs each player's type, and doing it by hand doesn't scale. Multinomial logistic regression outputs the *chance* a player is each type, then picks the most likely. Nothing to run here — just the goal.

## Step 2 — Meet the data

**What:** one row per player, one column per clue. Let's look at the shape and the first few rows.

**Why:** you can't sort what you don't understand — know your rows and columns first.

In [ ]:
# Step 2 — what do we have? (see note above)
print('Rows (players):', df.shape[0], ' | Columns:', df.shape[1])
df.head()

In [ ]:
# Step 2 — choose the numeric clues (features) the model gets to look at
FEATURES = ['registered_days_ago','lifetime_hands','avg_session_minutes','sessions_last_30d',
            'vpip','pfr','aggression_factor','avg_pot_size_bb','promo_redemptions_30d']
FEATURES = [c for c in FEATURES if c in df.columns]
print('Features we will use:', FEATURES)

## Step 3 — Explore the clues

**What:** look at the spread of each clue and spot anything weird (blanks, impossible values).

**Why:** surprises here quietly mislead the model later. Catch them now, cheaply.

In [ ]:
# Step 3 — quick numbers + a histogram per clue
display(df[FEATURES].describe().T[['mean','min','max']])
df[FEATURES].hist(figsize=(12,7), bins=20, color='#46c98b', edgecolor='#0c1810')
plt.tight_layout(); plt.show()

## Step 4 — Fix missing & odd values

**What:** fill any blanks with a sensible stand-in (the column's median) so every row is usable.

**Why:** the math can't run on blanks, and one bad value skews everything.

In [ ]:
# Step 4 — count blanks, then fill them with the median (see note above)
print('Missing values per clue:\n', df[FEATURES].isna().sum())
df[FEATURES] = df[FEATURES].fillna(df[FEATURES].median())
print('Missing after fill:', int(df[FEATURES].isna().sum().sum()))

## Step 5 — Put the clues on the same scale

**What:** rescale every clue to "how far from average" so big-number clues (lifetime_hands) don't drown out small ones (vpip).

**Why:** otherwise the model thinks a clue matters just because its numbers are bigger. This makes them compete fairly.

In [ ]:
# Step 5 — standardize the clues (mean 0, spread 1)
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])
y = df['archetype'].values
print('Scaled clue matrix:', X.shape)

## Step 6 — Drop redundant clues

**What:** if two clues say almost the same thing (e.g. two aggression measures), keep one. We use a correlation check here.

**Why:** overlapping clues confuse the model and blur the "why." Fewer, distinct clues = a clearer, steadier model.

In [ ]:
# Step 6 — show how clues overlap; pairs near +/-1 are 'twins' to consider dropping
corr = pd.DataFrame(X, columns=FEATURES).corr()
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(corr, cmap='BrBG', vmin=-1, vmax=1)
ax.set_xticks(range(len(FEATURES))); ax.set_xticklabels(FEATURES, rotation=90)
ax.set_yticks(range(len(FEATURES))); ax.set_yticklabels(FEATURES)
fig.colorbar(im); plt.title('Clue overlap (dark = strong overlap)'); plt.tight_layout(); plt.show()
# Tip: if two clues are above ~0.85, keep the one that's easier to explain.

## Step 7 — Pick a baseline type *(special to multinomial)*

**What:** a multinomial model describes every type *relative to one chosen "default."* We'll use `recreational` as the baseline, so results read as "compared to a recreational player…"

**Why:** the model needs an anchor to compare against; a common, ordinary type makes the weights easy to read. *(A yes/no model skips this — see the OvR notebook.)*

In [ ]:
# Step 7 — note our baseline. (scikit-learn uses a symmetric 'softmax' form, so the
# baseline is mostly an interpretation aid; statsmodels MNLogit lets you set it explicitly.)
BASELINE = 'recreational'
print('Reading the results as: compared to a', BASELINE, 'player...')

## Step 8 — Split: study set vs quiz set

**What:** teach on ~80% of players, hide ~20% for an honest final grade.

**Why:** we want proof it works on players it never saw — not that it memorized.

In [ ]:
# Step 8 — split (stratify keeps each type represented in both piles)
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
print(f'Study set: {len(y_tr)} players  |  Quiz set: {len(y_te)} players')

## Step 9 — Train the model

**What:** run the algorithm on the study set; it finds a weight for each clue (per type).

**Why:** this is the actual learning — turning examples into a reusable rule.

In [ ]:
# Step 9 — fit one multinomial logistic regression over all 9 types
from sklearn.linear_model import LogisticRegression
model = LogisticRegression(max_iter=2000, C=1.0)   # C controls regularization (clue-trimming)
model.fit(X_tr, y_tr)
print('Trained. Types the model knows:', list(model.classes_))

## Step 10 — Read what it learned (the reason codes)

**What:** each clue gets a weight per type. Big positive = strong evidence *for* that type.

**Why:** this is the payoff — the model's reasoning in plain numbers, usable as reason codes.

In [ ]:
# Step 10 — show the weights, then the top clues pushing toward one example type
weights = pd.DataFrame(model.coef_, index=model.classes_, columns=FEATURES).round(2)
display(weights)
ex = 'aggressive_predatory' if 'aggressive_predatory' in weights.index else weights.index[0]
print(f'\nTop clues pushing TOWARD "{ex}":')
print(weights.loc[ex].sort_values(ascending=False).head(4))

## Step 11 — How often is it right?

**What:** let the model guess the hidden quiz players and compare to the truth.

**Why:** the honest report card, on players it never studied.

In [ ]:
# Step 11 — accuracy on the hidden quiz set
from sklearn.metrics import accuracy_score
pred = model.predict(X_te)
print(f'Accuracy on quiz set: {accuracy_score(y_te, pred):.0%}')

## Step 12 — Where does it get confused?

**What:** a confusion matrix shows which types get mixed up (the off-diagonal cells).

**Why:** one accuracy number hides patterns; knowing *where* it fails tells you what to fix — and whether to trust it.

In [ ]:
# Step 12 — confusion matrix (green diagonal = correct)
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(8,7))
ConfusionMatrixDisplay.from_predictions(y_te, pred, ax=ax, xticks_rotation=90, cmap='Greens')
plt.title('Where the model agrees / disagrees with the truth'); plt.tight_layout(); plt.show()

---
## Done

You built a multinomial classifier end to end: it labels any player, you have an honest score, a map of its weak spots, and readable reasons for each call.

**Want the scorecard-style version** (WoE binning, Information Value, points, KS) with a separate model per type? See the **[One-vs-Rest notebook](ovr-notebook.ipynb)** and the **[MN vs OvR explainer](mn-vs-ovr.html)**.